# ABC2Vec — Notebook 07: Retrieval Evaluation (FAISS)

**Core evaluation:** Same-tune-family retrieval using pre-trained ABC2Vec embeddings.

**Pipeline:**
1. Load trained ABC2Vec model
2. Encode all benchmark tunes → embedding vectors
3. Build FAISS index over corpus
4. For each query, retrieve top-K nearest neighbors
5. Compute MRR, Recall@1, Recall@5, Recall@10, MAP

Also compares against baselines: random, TF-IDF, edit distance.

In [1]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

Device: mps


## 7.1 Load Model and Benchmark Data

In [2]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

# Load vocab
vocab_path = PROCESSED_DIR / 'vocab.json'
if vocab_path.exists():
    vocab = ABCVocabulary.load(vocab_path)
    print(f"Vocabulary: {vocab.size} tokens")
else:
    print("WARNING: No vocabulary found. Run notebook 02 first.")
    vocab = None

# Load patchifier
patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

# Load model config (saved during pretraining — guaranteed to match checkpoint)
config_path = PROCESSED_DIR / 'model_config.json'
if config_path.exists():
    config = ABC2VecConfig.load(config_path)
    print(f"Loaded model config from {config_path}")
else:
    # Fallback: construct manually
    config = ABC2VecConfig(
        vocab_size=vocab.size,
        max_bar_length=64,
        max_bars=64,
        d_model=256,
        n_heads=8,
        n_layers=6,
        d_ff=1024,
        d_embed=128,
        dropout=0.1,
    )

model = ABC2VecModel(config).to(device)

# Load best checkpoint
ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint: epoch {checkpoint.get('epoch', '?')}, "
          f"val_loss {checkpoint.get('loss', '?'):.4f}")
else:
    print("WARNING: No checkpoint found. Using untrained model.")

model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load benchmark
queries_df = pd.read_parquet(BENCHMARK_DIR / 'retrieval_queries.parquet')
corpus_df = pd.read_parquet(BENCHMARK_DIR / 'retrieval_corpus.parquet')
print(f"\nBenchmark: {len(queries_df)} queries, {len(corpus_df)} corpus entries")


Vocabulary: 98 tokens
Loaded model config from /Volumes/LLModels/ABC2VEC/data/processed/model_config.json
Loaded checkpoint: epoch 19, val_loss 2.4057
Model parameters: 4,985,698

Benchmark: 2349 queries, 14101 corpus entries


## 7.2 Encode All Tunes

In [3]:
import gc

@torch.no_grad()
def encode_tunes(abc_texts: list[str], model, patchifier, device,
                 batch_size: int = 64) -> np.ndarray:
    """
    Encode a list of ABC notation strings into embedding vectors.
    Returns: numpy array of shape (N, d_embed)
    """
    model.eval()
    all_embeddings = []
    
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch_texts = abc_texts[i:i+batch_size]
        
        bar_ids_list = []
        char_mask_list = []
        bar_mask_list = []
        
        for text in batch_texts:
            patch = patchifier.patchify(text)
            bar_ids_list.append(patch['bar_indices'])
            char_mask_list.append(patch['char_mask'])
            bar_mask_list.append(patch['bar_mask'])
        
        bar_ids_tensor   = torch.stack(bar_ids_list).to(device)
        char_mask_tensor = torch.stack(char_mask_list).to(device)
        bar_mask_tensor  = torch.stack(bar_mask_list).to(device)
        
        embeddings = model.get_embedding(bar_ids_tensor, char_mask_tensor, bar_mask_tensor)
        all_embeddings.append(embeddings.cpu().numpy())
    
    return np.concatenate(all_embeddings, axis=0)

# Encode queries and corpus
print("Encoding queries...")
query_embeddings = encode_tunes(queries_df['abc_body'].tolist(),
                                 model, patchifier, device)
print(f"Query embeddings: {query_embeddings.shape}")

print("\nEncoding corpus...")
corpus_embeddings = encode_tunes(corpus_df['abc_body'].tolist(),
                                  model, patchifier, device)
print(f"Corpus embeddings: {corpus_embeddings.shape}")

# Save embeddings — also serves as a cache if the kernel restarts
np.save(RESULTS_DIR / 'query_embeddings.npy', query_embeddings)
np.save(RESULTS_DIR / 'corpus_embeddings.npy', corpus_embeddings)
print("Embeddings saved.")

# Free model from device memory — it's no longer needed for the evaluation steps
# and keeping it loaded reduces headroom for the baseline computations.
del model
if device.type == 'mps':
    torch.mps.empty_cache()
elif device.type == 'cuda':
    torch.cuda.empty_cache()
gc.collect()
print("Model freed from memory.")


Encoding queries...


Encoding:   0%|          | 0/37 [00:00<?, ?it/s]

Query embeddings: (2349, 128)

Encoding corpus...


Encoding:   0%|          | 0/221 [00:00<?, ?it/s]

Corpus embeddings: (14101, 128)
Embeddings saved.
Model freed from memory.


## 7.3 Build FAISS Index and Retrieve

In [ ]:
def build_faiss_index(embeddings: np.ndarray, use_cosine: bool = True):
    """Build a FAISS index for nearest-neighbor search."""
    d = embeddings.shape[1]
    if use_cosine:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings_norm = embeddings / np.maximum(norms, 1e-8)
        index = faiss.IndexFlatIP(d)
        index.add(embeddings_norm.astype(np.float32))
        return index, embeddings_norm
    else:
        index = faiss.IndexFlatL2(d)
        index.add(embeddings.astype(np.float32))
        return index, embeddings

K_VALUES = [1, 5, 10, 20, 50]
MAX_K = max(K_VALUES)

# Load embeddings: use in-memory arrays if present, otherwise fall back to disk cache.
_q_path = RESULTS_DIR / 'query_embeddings.npy'
_c_path = RESULTS_DIR / 'corpus_embeddings.npy'

if 'corpus_embeddings' not in dir() or corpus_embeddings is None:
    if _q_path.exists() and _c_path.exists():
        print("Loading saved embeddings from disk...")
        query_embeddings  = np.load(_q_path)
        corpus_embeddings = np.load(_c_path)
        print(f"Loaded — queries: {query_embeddings.shape}, corpus: {corpus_embeddings.shape}")
    else:
        raise RuntimeError(
            "Embeddings not found in memory or on disk.\n"
            "Please run cells 7.1 and 7.2 first to encode the benchmark tunes."
        )

# Build FAISS index over corpus
index, corpus_norm = build_faiss_index(corpus_embeddings, use_cosine=True)
print(f"FAISS index: {index.ntotal} vectors, dim={corpus_embeddings.shape[1]}")

# Normalize query embeddings for cosine search
query_norms = np.linalg.norm(query_embeddings, axis=1, keepdims=True)
query_norm  = query_embeddings / np.maximum(query_norms, 1e-8)

# Retrieve top-K
distances, indices = index.search(query_norm.astype(np.float32), MAX_K)
print(f"Retrieved top-{MAX_K} for {len(query_norm)} queries")


## 7.4 Compute Retrieval Metrics

In [ ]:
def compute_retrieval_metrics(query_families, corpus_families, retrieved_indices,
                               k_values=[1, 5, 10, 20, 50]):
    """
    Compute retrieval metrics: MRR, Recall@K, MAP.
    
    Args:
        query_families: array of family IDs for queries
        corpus_families: array of family IDs for corpus
        retrieved_indices: (N_queries, MAX_K) array of retrieved corpus indices
    """
    n_queries = len(query_families)
    max_k = retrieved_indices.shape[1]
    
    mrr_sum = 0.0
    recall_at_k = {k: 0.0 for k in k_values}
    map_sum = 0.0
    
    per_query_results = []
    
    for i in range(n_queries):
        q_family = query_families[i]
        retrieved_families = corpus_families[retrieved_indices[i]]
        
        # Total number of relevant items in corpus
        n_relevant = np.sum(corpus_families == q_family)
        
        if n_relevant == 0:
            continue
        
        # Relevance mask
        is_relevant = (retrieved_families == q_family)
        
        # MRR: rank of first relevant result
        relevant_positions = np.where(is_relevant)[0]
        if len(relevant_positions) > 0:
            first_rank = relevant_positions[0] + 1  # 1-indexed
            mrr_sum += 1.0 / first_rank
        
        # Recall@K
        for k in k_values:
            hits_at_k = np.sum(is_relevant[:k])
            recall_at_k[k] += hits_at_k / min(n_relevant, k)
        
        # AP (Average Precision)
        ap = 0.0
        n_hits = 0
        for j in range(max_k):
            if is_relevant[j]:
                n_hits += 1
                ap += n_hits / (j + 1)
        if n_relevant > 0:
            ap /= min(n_relevant, max_k)
        map_sum += ap
        
        per_query_results.append({
            'family_id': q_family,
            'n_relevant': n_relevant,
            'first_relevant_rank': int(relevant_positions[0] + 1) if len(relevant_positions) > 0 else -1,
            'ap': ap,
        })
    
    n_valid = len(per_query_results)
    metrics = {
        'MRR': mrr_sum / n_valid,
        'MAP': map_sum / n_valid,
        'n_queries': n_valid,
    }
    for k in k_values:
        metrics[f'Recall@{k}'] = recall_at_k[k] / n_valid
    
    return metrics, per_query_results

# Compute metrics
query_families = queries_df['family_id'].values
corpus_families = corpus_df['family_id'].values

metrics, per_query = compute_retrieval_metrics(
    query_families, corpus_families, indices, K_VALUES
)

print("=" * 50)
print("ABC2Vec Retrieval Results")
print("=" * 50)
print(f"  MRR:        {metrics['MRR']:.4f}")
print(f"  MAP:        {metrics['MAP']:.4f}")
for k in K_VALUES:
    print(f"  Recall@{k:<3d}: {metrics[f'Recall@{k}']:.4f}")
print(f"  N queries:  {metrics['n_queries']}")

## 7.5 Baselines for Comparison

In [ ]:
import gc
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

# Force-release any lingering MPS/CUDA memory from earlier cells before
# the baseline computations allocate their own buffers.
if torch.backends.mps.is_available():
    torch.mps.synchronize()
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
gc.collect()

# ─── Baseline 1: Random ───
np.random.seed(42)
random_indices = np.array([
    np.random.permutation(len(corpus_df))[:MAX_K]
    for _ in range(len(queries_df))
])
random_metrics, _ = compute_retrieval_metrics(
    query_families, corpus_families, random_indices, K_VALUES
)
print(f"Random done.  MRR={random_metrics['MRR']:.4f}")

# ─── Baseline 2: TF-IDF (char n-grams), batched sparse matmul ───
print("Computing TF-IDF baseline...")
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=3000)
corpus_tfidf = tfidf.fit_transform(corpus_df['abc_body'].tolist())
query_tfidf  = tfidf.transform(queries_df['abc_body'].tolist())

BATCH = 256
tfidf_indices = np.empty((len(queries_df), MAX_K), dtype=np.int32)
for i in range(0, len(queries_df), BATCH):
    q_batch = query_tfidf[i:i+BATCH]
    sim = (q_batch @ corpus_tfidf.T).toarray()
    tfidf_indices[i:i+BATCH] = np.argsort(-sim, axis=1)[:, :MAX_K]

del corpus_tfidf, query_tfidf
gc.collect()

tfidf_metrics, _ = compute_retrieval_metrics(
    query_families, corpus_families, tfidf_indices, K_VALUES
)
print(f"TF-IDF done.  MRR={tfidf_metrics['MRR']:.4f}")

# ─── Baseline 3: Char trigram cosine (binary bag-of-ngrams via FAISS) ───
print("Computing char-Jaccard baseline...")
jac_vec = TfidfVectorizer(
    analyzer='char', ngram_range=(3, 3),
    max_features=2000, binary=True, use_idf=False, norm='l2',
)
corpus_jac = jac_vec.fit_transform(corpus_df['abc_body'].tolist()).toarray().astype(np.float32)
query_jac  = jac_vec.transform(queries_df['abc_body'].tolist()).toarray().astype(np.float32)

faiss.normalize_L2(corpus_jac)
faiss.normalize_L2(query_jac)
jac_index = faiss.IndexFlatIP(corpus_jac.shape[1])
jac_index.add(corpus_jac)
_, edit_indices = jac_index.search(query_jac, MAX_K)

del corpus_jac, query_jac, jac_index
gc.collect()

edit_metrics, _ = compute_retrieval_metrics(
    query_families, corpus_families, edit_indices, K_VALUES
)
print(f"Char-Jaccard done. MRR={edit_metrics['MRR']:.4f}")


## 7.6 Compare All Methods

In [ ]:
# Build comparison table
comparison = {
    'Method': ['Random', 'Char Jaccard', 'TF-IDF (char n-grams)', 'ABC2Vec (ours)'],
    'MRR': [random_metrics['MRR'], edit_metrics['MRR'], tfidf_metrics['MRR'], metrics['MRR']],
    'MAP': [random_metrics['MAP'], edit_metrics['MAP'], tfidf_metrics['MAP'], metrics['MAP']],
}
for k in K_VALUES:
    comparison[f'R@{k}'] = [
        random_metrics[f'Recall@{k}'],
        edit_metrics[f'Recall@{k}'],
        tfidf_metrics[f'Recall@{k}'],
        metrics[f'Recall@{k}'],
    ]

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Save
comparison_df.to_csv(RESULTS_DIR / 'retrieval_comparison.csv', index=False)

# ─── Plot ───
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = comparison_df['Method'].tolist()
colors = ['#bbb', '#e6a44a', '#5ba8e6', '#e6445a']

# MRR
axes[0].barh(methods, comparison_df['MRR'], color=colors)
axes[0].set_xlabel('MRR')
axes[0].set_title('Mean Reciprocal Rank')
axes[0].set_xlim(0, 1)

# MAP
axes[1].barh(methods, comparison_df['MAP'], color=colors)
axes[1].set_xlabel('MAP')
axes[1].set_title('Mean Average Precision')
axes[1].set_xlim(0, 1)

# Recall@K curves
for i, method in enumerate(methods):
    recall_vals = [comparison_df[f'R@{k}'].iloc[i] for k in K_VALUES]
    axes[2].plot(K_VALUES, recall_vals, 'o-', label=method, color=colors[i], linewidth=2)
axes[2].set_xlabel('K')
axes[2].set_ylabel('Recall@K')
axes[2].set_title('Recall at K')
axes[2].legend(loc='lower right')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'retrieval_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved to {RESULTS_DIR}")

## 7.7 Error Analysis

In [ ]:
# Analyze which queries are hardest
per_query_df = pd.DataFrame(per_query)

# Hardest queries (first relevant result at high rank)
hardest = per_query_df[per_query_df['first_relevant_rank'] > 0].nlargest(10, 'first_relevant_rank')
print("Top 10 hardest queries (highest first-relevant rank):")
for _, row in hardest.iterrows():
    q_entry = queries_df[queries_df['family_id'] == row['family_id']].iloc[0]
    print(f"  Family {row['family_id']}: '{q_entry['tune_name']}' "
          f"(type={q_entry['tune_type']}, rank={row['first_relevant_rank']}, "
          f"n_relevant={row['n_relevant']})")

# Distribution of first-relevant rank
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ranks = per_query_df[per_query_df['first_relevant_rank'] > 0]['first_relevant_rank']
axes[0].hist(ranks, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Rank of first relevant result')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of First Relevant Rank')
axes[0].axvline(x=ranks.median(), color='red', linestyle='--', label=f'Median={ranks.median():.0f}')
axes[0].legend()

# AP by tune type
per_query_df = per_query_df.merge(
    queries_df[['family_id', 'tune_type']].drop_duplicates(),
    on='family_id', how='left'
)
type_ap = per_query_df.groupby('tune_type')['ap'].mean().sort_values(ascending=True)
type_ap.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Mean AP')
axes[1].set_title('Average Precision by Tune Type')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'retrieval_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.8 Qualitative Examples

In [ ]:
# Show some query + top-5 retrieved examples
print("=" * 60)
print("Qualitative Retrieval Examples")
print("=" * 60)

n_examples = 5
np.random.seed(123)
example_indices = np.random.choice(len(queries_df), n_examples, replace=False)

for idx in example_indices:
    q = queries_df.iloc[idx]
    print(f"\n--- Query: '{q['tune_name']}' (type={q['tune_type']}, family={q['family_id']}) ---")
    print(f"ABC: {q['abc_body'][:120]}...")
    print(f"\nTop-5 Retrieved:")
    
    for rank in range(5):
        c_idx = indices[idx, rank]
        c = corpus_df.iloc[c_idx]
        is_match = '✓' if c['family_id'] == q['family_id'] else '✗'
        print(f"  {rank+1}. [{is_match}] '{c['tune_name']}' "
              f"(type={c['tune_type']}, family={c['family_id']}, "
              f"sim={distances[idx, rank]:.4f})")
        print(f"     ABC: {c['abc_body'][:100]}...")
    print()